To use this code effectively, on the Tim Horotons app, there is a game called Tims NHL Challenge. The details about the game is in the app, so if you want to find out more check it out. But what you need to know is that there is no risk involved. This program has you type in each of three lists of NHL players and it outputs who is most likely to score using math and player stats. Don't forget to seperate each player with a comma. You take these three players and use them as your picks to get you the chance to win free rewards points and coffee. As of 30 of March 2026, this program has had a 75% accuracy of getting at least of the players correct. Good luck!!

In [ ]:
!pip -q install requests python-dateutil
import requests
import re
from datetime import datetime, timedelta, timezone
from dateutil import parser
!pip -q install ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.3 MB/s eta 0:00:00


In [ ]:


base = "https://api-web.nhle.com"
headers = {"User-Agent": "tims-nhl-goal-picker/1.0"}


team_abb = [
    "ANA","BOS","BUF","CAR","CBJ","CGY",
    "CHI","COL","DAL","DET","EDM","FLA",
    "LAK","MIN","MTL","NJD","NSH","NYI",
    "NYR","OTT","PHI","PIT","SJS","SEA",
    "STL","TBL","TOR","UTA","VAN","VGK",
                "WPG","WSH"
]

def api_get(path):
    url = f"{base}{path}"
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    return r.json()

def normalize_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z\s\-\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def today_date_str_local():
    return datetime.now().date().isoformat()

def fetch_all_rosters():
    """
    Build a name->playerId index by pulling current rosters for all teams:
    /v1/roster/{team}/current  [1](https://github.com/Zmalski/NHL-API-Reference)
    """
    index = {}
    for team in team_abb:
        try:
            data = api_get(f"/v1/roster/{team}/current")
        except Exception:
            continue


        for group_key, players in data.items():
            if not isinstance(players, list):
                continue
            for p in players:
                pid = p.get("id") or p.get("playerId")
                if not pid:
                    continue

                first = (p.get("firstName", {}) or {}).get("default") if isinstance(p.get("firstName"), dict) else p.get("firstName")
                last  = (p.get("lastName", {}) or {}).get("default")  if isinstance(p.get("lastName"), dict)  else p.get("lastName")
                name  = p.get("name", {}).get("default") if isinstance(p.get("name"), dict) else p.get("name")

                if first and last:
                    full = f"{first} {last}"
                elif name:
                    full = name
                else:
                    continue

                key = normalize_name(full)
                index.setdefault(key, []).append({
                    "playerId": int(pid),
                    "displayName": full,
                    "team": team
                })
    return index

def resolve_player(name: str, roster_index):
    """
    Resolve a typed player name to a single playerId.
    If ambiguous (rare), pick the first; you can upgrade to interactive selection.
    """
    key = normalize_name(name)
    if key in roster_index:
        return roster_index[key][0]


    candidates = []
    for k, lst in roster_index.items():
        if key in k or k in key:
            candidates.extend(lst)

    if candidates:

        candidates.sort(key=lambda c: abs(len(normalize_name(c["displayName"])) - len(key)))
        return candidates[0]

    return None

def fetch_todays_games(date_str):
    """
    /v1/schedule/{date} returns games for the date. [1](https://github.com/Zmalski/NHL-API-Reference)
    We'll map team abbrev -> (gameId, opponent abbrev, isHome).
    """
    sched = api_get(f"/v1/schedule/{date_str}")
    team_to_game = {}



    def walk(obj):
        if isinstance(obj, dict):

            if "homeTeam" in obj and "awayTeam" in obj and ("id" in obj or "gameId" in obj):
                gid = obj.get("id") or obj.get("gameId")
                home = (obj["homeTeam"] or {}).get("abbrev")
                away = (obj["awayTeam"] or {}).get("abbrev")
                if gid and home and away:
                    team_to_game[home] = (int(gid), away, True)
                    team_to_game[away] = (int(gid), away, False)

            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for x in obj:
                walk(x)

    walk(sched)
    return team_to_game

def fetch_player_game_log_now(player_id):
    """
    /v1/player/{player}/game-log/now  [1](https://github.com/Zmalski/NHL-API-Reference)
    """
    return api_get(f"/v1/player/{player_id}/game-log/now")

def extract_recent_features(game_log_json, last_n=10):
    """
    Extract recent form features from game log:
    - goals_lastN
    - shots_lastN (sog)
    - avg_toi_lastN (minutes)
    - last_game_date
    """
    games = game_log_json.get("gameLog", []) or game_log_json.get("games", []) or []
    if not games:
        return {
            "games_used": 0,
            "goals": 0,
            "shots": 0,
            "avg_toi": 0.0,
            "last_game_date": None
        }

    games = games[:last_n]
    goals = 0
    shots = 0
    toi_minutes = []

    last_date = None

    for g in games:
        goals += int(g.get("goals", 0) or 0)
        shots += int(g.get("shots", 0) or g.get("sog", 0) or 0)

        toi = g.get("toi") or g.get("timeOnIce") or None
        if isinstance(toi, str) and ":" in toi:
            mm, ss = toi.split(":")
            toi_minutes.append(int(mm) + int(ss)/60.0)
        elif isinstance(toi, (int, float)):
            toi_minutes.append(float(toi))


        d = g.get("gameDate") or g.get("date")
        if d and not last_date:
            try:
                last_date = parser.parse(d).date()
            except Exception:
                pass

    avg_toi = sum(toi_minutes)/len(toi_minutes) if toi_minutes else 0.0
    return {
        "games_used": len(games),
        "goals": goals,
        "shots": shots,
        "avg_toi": avg_toi,
        "last_game_date": last_date
    }

def fetch_team_week_schedule(team):
    """
    /v1/club-schedule/{team}/week/now  [1](https://github.com/Zmalski/NHL-API-Reference)
    Used to compute fatigue.
    """
    return api_get(f"/v1/club-schedule/{team}/week/now")

def compute_team_fatigue(team, date_str):
    """
    Fatigue proxy:
    - games_last_7: count of games in last 7 days (including today if applicable)
    - days_rest: days since last game before today (0 = played yesterday)
    """
    target = datetime.fromisoformat(date_str).date()
    start = target - timedelta(days=7)

    sched = fetch_team_week_schedule(team)


    dates = []

    def walk(obj):
        if isinstance(obj, dict):
            if "gameDate" in obj and ("id" in obj or "gameId" in obj):
                try:
                    d = parser.parse(obj["gameDate"]).date()
                    dates.append(d)
                except Exception:
                    pass
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for x in obj:
                walk(x)

    walk(sched)

    dates = sorted(set(dates))
    games_last_7 = sum(1 for d in dates if start <= d <= target)


    prev_games = [d for d in dates if d < target]
    days_rest = (target - prev_games[-1]).days if prev_games else 7

    return {"games_last_7": games_last_7, "days_rest": days_rest}

def fetch_club_stats_now(team):
    """
    /v1/club-stats/{team}/now  [1](https://github.com/Zmalski/NHL-API-Reference)
    We’ll use opponent defense proxies (goalsAgainstPerGame if present).
    """
    return api_get(f"/v1/club-stats/{team}/now")

def extract_defense_proxy(club_stats_json):
    """
    Try common keys; fall back to neutral value if missing.
    Lower GA/GP = tougher matchup -> reduce scoring likelihood.
    """

    for k in ["goalsAgainstPerGame", "goalsAgainstAvg", "goalsAgainstPerGamePlayed", "gaPerGame"]:
        v = club_stats_json.get(k)
        if isinstance(v, (int, float)):
            return float(v)


    if "teamStats" in club_stats_json and isinstance(club_stats_json["teamStats"], dict):
        ts = club_stats_json["teamStats"]
        for k in ["goalsAgainstPerGame", "goalsAgainstAvg"]:
            v = ts.get(k)
            if isinstance(v, (int, float)):
                return float(v)

    return 3.2

def score_player(player, date_str, team_to_game):
    """
    Produce a goal-likelihood SCORE (not a true probability) using:
    - recent goals & shots (form)
    - usage proxy (avg TOI)
    - fatigue (days rest, games last 7)
    - matchup (opponent defensive proxy)
    """
    team = player["team"]
    plays_today = team in team_to_game

    if not plays_today:
        return {
            "score": -999,
            "reason": "No game today",
            "details": {}
        }

    game_id, opp, is_home = team_to_game[team]
# forumals

    gl = fetch_player_game_log_now(player["playerId"])
    form = extract_recent_features(gl, last_n=6)


    fat = compute_team_fatigue(team, date_str)


    opp_stats = fetch_club_stats_now(opp)
    opp_ga = extract_defense_proxy(opp_stats)


    shots_pg = (form["shots"] / form["games_used"]) if form["games_used"] else 0
    goals_pg = (form["goals"] / form["games_used"]) if form["games_used"] else 0
    toi = form["avg_toi"]


    fatigue_pen = 0.0
    if fat["days_rest"] <= 1:
        fatigue_pen += 0.25
    fatigue_pen += max(0, fat["games_last_7"] - 3) * 0.08



    matchup_adj = (opp_ga - 3.2) * 0.25

    home_adj = 0.05 if is_home else 0.0

    raw = (
        1.25 * shots_pg +
        0.90 * goals_pg +
        0.03 * toi +
        matchup_adj +
        home_adj
        - fatigue_pen
    )

    return {
        "score": raw,
        "reason": "OK",
        "details": {
            "gameId": game_id,
            "opponent": opp,
            "home": is_home,
            "shots_pg_last6": round(shots_pg, 3),
            "goals_pg_last6": round(goals_pg, 3),
            "avg_toi_last6": round(toi, 2),
            "days_rest": fat["days_rest"],
            "games_last_6": fat["games_last_7"],
            "opp_goals_against_per_game_proxy": round(opp_ga, 3),
            "fatigue_penalty": round(fatigue_pen, 3),
            "matchup_adjustment": round(matchup_adj, 3),
            "home_adjustment": round(home_adj, 3)
        }
    }

def pick_best_from_list(names, roster_index, date_str, team_to_game):
    scored = []
    unresolved = []

    for name in names:
        if not name.strip():
            continue
        p = resolve_player(name, roster_index)
        if not p:
            unresolved.append(name)
            continue
        s = score_player(p, date_str, team_to_game)
        scored.append((p, s))

    scored.sort(key=lambda x: x[1]["score"], reverse=True)
    return scored, unresolved

def parse_input_list(s: str):

    parts = []
    for chunk in s.replace("\n", ",").split(","):
        t = chunk.strip()
        if t:
            parts.append(t)
    return parts

def main():
    date_str = today_date_str_local()

    print("Building roster index (this can take ~10-30 seconds the first time)...")
    roster_index = fetch_all_rosters()
    print(f"Loaded {len(roster_index)} unique normalized player names from current rosters.")

    print(f"\nFetching today's schedule ({date_str})...")
    team_to_game = fetch_todays_games(date_str)

    print("\nEnter List 1 players (comma or newline separated), then press Enter twice:")
    list1 = []
    while True:
        line = input()
        if not line.strip():
            break
        list1.append(line)
    list1 = parse_input_list("\n".join(list1))

    print("\nEnter List 2 players (comma or newline separated), then press Enter twice:")
    list2 = []
    while True:
        line = input()
        if not line.strip():
            break
        list2.append(line)
    list2 = parse_input_list("\n".join(list2))

    print("\nEnter List 3 players (comma or newline separated), then press Enter twice:")
    list3 = []
    while True:
        line = input()
        if not line.strip():
            break
        list3.append(line)
    list3 = parse_input_list("\n".join(list3))

    for i, lst in enumerate([list1, list2, list3], start=1):
        print(f"\n=== RESULTS FOR LIST {i} ===")
        scored, unresolved = pick_best_from_list(lst, roster_index, date_str, team_to_game)

        if unresolved:
            print("Unresolved names (not found on current rosters):")
            for u in unresolved:
                print(f"  - {u}")

        if not scored:
            print("No valid players to score.")
            continue

        best_p, best_s = scored[0]
        print(f"\nTop pick: {best_p['displayName']} ({best_p['team']})")
        print(f"Score: {best_s['score']:.3f}")
        print("Breakdown:")
        for k, v in best_s["details"].items():
            print(f"  {k}: {v}")

        print("\nOther ranked options:")
        for p, s in scored[1:]:
            print(f"  - {p['displayName']} ({p['team']}): {s['score']:.3f}")

if __name__ == "__main__":
    main()